# Part 2 – Linear Regression, Ridge & Decision Tree (Nicoll)
Run from inside `1-challenge/`. Same 5-fold CV / RMSE protocol as the rest of the team notebook.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import KFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

SEED = 42
train = pd.read_csv("data/challenge1_train.csv")
test  = pd.read_csv("data/challenge1_test.csv")
print(train.shape, test.shape)
train.head()

(7398, 10) (2467, 9)


,ID,price,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state
0,2,925,1.0,0,116,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA
1,3,2475,1.0,0,130,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY
2,5,1695,1.0,0,190,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA
3,6,1560,1.0,1,200,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC
4,7,1560,1.0,1,200,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC


## 2. Preprocessing & validation strategy
* **Preprocessing lives inside sklearn Pipelines**, so scaling/encoding are fit only on the training folds (no leakage).
* Linear model: standardize numerics + one-hot categoricals. Tree models: no scaling needed.
* **Validation: 5-fold shuffled KFold (seed 42) on the training data only**, used for every model, every grid search and the final choice. Metric: RMSE on the original dollar scale. We report train RMSE vs CV RMSE to diagnose over/underfitting.
* Test set is never touched until the final prediction.

In [2]:
X = train.drop(columns=["ID","price","category"]); y = train["price"]
X_test = test.drop(columns=["ID","category"])
num = ["bathrooms","bedrooms","square_feet","latitude","longitude"]; cat = ["has_photo","state"]
cv = KFold(5, shuffle=True, random_state=SEED)

def prep(scale):
    return ColumnTransformer([("num", StandardScaler() if scale else "passthrough", num),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])

def evaluate(name, model, X=X):
    r = cross_validate(model, X, y, cv=cv, scoring="neg_root_mean_squared_error", return_train_score=True, n_jobs=-1)
    return dict(model=name, train_RMSE=-r["train_score"].mean(), cv_RMSE=-r["test_score"].mean(), cv_std=r["test_score"].std())

## Baselines (default hyperparameters)

In [3]:
from sklearn.dummy import DummyRegressor
base = {
 "Mean baseline (dummy)": DummyRegressor(),
 "Linear Regression": Pipeline([("p",prep(True)),("m",LinearRegression())]),
 "Decision Tree":     Pipeline([("p",prep(False)),("m",DecisionTreeRegressor(random_state=SEED))]),
}
res = pd.DataFrame([evaluate(k,v) for k,v in base.items()]).round(1); res

,model,train_RMSE,cv_RMSE,cv_std
0,Mean baseline (dummy),821.4,820.7,41.8
1,Linear Regression,521.0,526.8,25.3
2,Decision Tree,40.7,556.0,52.5


Default Decision Tree: tiny train RMSE vs much larger CV RMSE → overfits. Linear Regression: train ≈ CV but high → underfits (non-linear price vs size/location).

## Feature engineering + log target

In [4]:
def add_feats(d):
    d = d.copy()
    d["log_sqft"] = np.log1p(d.square_feet)
    d["total_rooms"] = d.bedrooms + d.bathrooms
    d["sqft_per_bed"] = d.square_feet / (d.bedrooms + 1)
    d["bath_per_bed"] = d.bathrooms / (d.bedrooms + 1)
    return d
FE = FunctionTransformer(add_feats)
num2 = num + ["log_sqft","total_rooms","sqft_per_bed","bath_per_bed"]
def prep2(scale):
    return ColumnTransformer([("num", StandardScaler() if scale else "passthrough", num2),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)])
def pipe(model, scale=False, logy=True):
    p = Pipeline([("fe",FE),("p",prep2(scale)),("m",model)])
    return TransformedTargetRegressor(p, func=np.log1p, inverse_func=np.expm1) if logy else p

In [5]:
print(pd.DataFrame([
  evaluate("LinReg raw target", pipe(LinearRegression(), scale=True, logy=False)),
  evaluate("LinReg + FE + log y", pipe(LinearRegression(), scale=True)),
]).round(1))

                 model  train_RMSE  cv_RMSE  cv_std
0    LinReg raw target       517.6    524.5    29.1
1  LinReg + FE + log y       506.3    512.2    29.6


## Grid Search (same 5-fold CV)

In [6]:
grids = {
 "Ridge": (pipe(Ridge(), scale=True), {"regressor__m__alpha":[0.1,1,10,100]}),
 "Decision Tree": (pipe(DecisionTreeRegressor(random_state=SEED)),
     {"regressor__m__max_depth":[4,6,8,10,14],"regressor__m__min_samples_leaf":[1,5,10,20]}),
}
rows=[]
for name,(est,g) in grids.items():
    gs = GridSearchCV(est, g, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1, return_train_score=True).fit(X,y)
    i=gs.best_index_
    rows.append(dict(model=name, train_RMSE=-gs.cv_results_["mean_train_score"][i], cv_RMSE=-gs.best_score_, cv_std=gs.cv_results_["std_test_score"][i], best_params={k.split("__")[-1]:v for k,v in gs.best_params_.items()}))
pd.DataFrame(rows).round(1)

,model,train_RMSE,cv_RMSE,cv_std,best_params
0,Ridge,506.5,512.3,29.7,{'alpha': 0.1}
1,Decision Tree,345.4,487.3,25.4,"{'max_depth': 14, 'min_samples_leaf': 5}"


Tuning the tree (depth + min leaf) reduces overfitting; Ridge barely changes linear regression, confirming the linear model is limited by bias, not variance. Both are clearly beaten by the ensembles in part 3.